# 00 Data and experimental design

## Learning outcome

Identify the source of each result, interpret one-based record IDs versus zero-based sample offsets, and explain why selective record exclusion is documented before diagnostic evaluation.

## Method and notation

The designated training and development validation files provide a single short time series each. Fault records 61 and 63 duplicate the beginning of the two designated records. They are retained in audit tables and excluded from group summaries. Group A/B refers to record index only.

Prerequisites are listed in the project README. This notebook reads the configured run directory. The tables and measurements below come from the recorded run or are recomputed in Python. Use `scripts/execute_notebooks.py --mode full --run-id NEW --data-dir PATH` to regenerate the shared pipeline.

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, display
from gearbox_dhon.data import find_input, load_record

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
run_id = os.getenv('GEARBOX_RUN_ID', 'development')
run = root / 'outputs' / 'runs' / run_id
candidate = Path(os.getenv('GEARBOX_DATA_DIR', str(root / 'data' / 'raw')))
data_dir = candidate if candidate.is_dir() and find_input(candidate, 'sigtrain') else None

def read(relative):
    path = run / relative
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def show_files(directory, patterns, limit):
    candidates = [p for p in sorted(directory.rglob('*'))
                  if p.suffix in {'.csv', '.json'} and any(k in p.stem.lower() for k in patterns)]
    for path in candidates[:limit]:
        print(path.relative_to(root))
        if path.suffix == '.csv':
            display(pd.read_csv(path).head(8))
        else:
            value = json.loads(path.read_text())
            print(json.dumps(value, indent=2)[:3600])

def show_figures(patterns, limit):
    candidates = [p for p in sorted((run / 'figures').rglob('*.png'))
                  if any(k in p.stem.lower() for k in patterns)]
    for path in candidates[:limit]:
        print(path.relative_to(root))
        display(Image(filename=str(path), width=690))


## From course rubric to result

This course project asks how accurately present gearbox vibration can be predicted from earlier observations under variable speed. An online predictor will see prior vibration and prior speed, then produce one sample before reading the corresponding target. MSE measures standardized one-step prediction. Batch CPU and GPU timing, matched by device and target mask, assess computational cost. Residual-based diagnostics are a separate objective because a predictor can reduce MSE by explaining some of a periodic fault pulse. The two supervised prediction inputs and two prospective blind inputs are distinct in the course brief. The PDFs and prior archives are evidence and historical comparisons.

The designated training and development validation files each contain 24,000 samples. The first timestamp is 0.0003125 s, the last is 7.5 s, and the nominal duration is 24,000/3,200 = 7.5 s. The saved sample rate is consistent with all differences. Each 10-second candidate contains 32,000 samples. Each of the 20 numbered parts is a 1×10 MATLAB struct array. The first element of part 007 is therefore record 61. Original IDs are one-based. Python ranges use zero-based indices and exclude the upper boundary.

Reading sample values checks both vibration and shaft speed at every aligned offset. A nearly identical speed profile is weak evidence of waveform duplication because the same profile was imposed repeatedly. Exact vibration and speed overlap links record 61 to course training and record 63 to course validation. The inference keeps both entire acquisitions outside the group comparison. Correlation screening across all full-length offsets and exact partial-marker scans document the search scope. The manifest flags all 200 candidates and preserves every record's original prediction/diagnostic rows when those stages are run.

## Scaling and validation design

MAT files provide standardized `x`, plus `xraw` and a fixed affine transform for the two short records. Recover the supplied pre-standardization signal via $x_{
m raw}=mu+sigma x$, with $mu=0.00262219$ and $sigma=0.42517293$ on the first channel. The numerical test matches within floating-point rounding. The file lacks a documented physical unit for `xraw`, so use squared pre-standardization units for an affine conversion. Dividing each fault record by its own standard deviation would remove amplitude evidence.

The data are one short fit series and one closely matched development series. The archived prediction scoring mask starts at index 1792 to give all models sufficient history. Selection uses three fixed held-out blocks of the designated training series, with a 1792-sample exclusion guard. Derived inputs with state or residual memory require their own support audit. A prediction at a validation target may use earlier observed vibration and speed under the causal one-step protocol. Fitting parameters on any held-out target remains disallowed. The run tracks the difference between proof of a software information cutoff and independent acquisition-level inference.

Chen et al. (2025) §5.1 describes a 20-second waveform with two distinct historical ranges. Fig. 13(c) shows a different speed phase from the course validation file. External results therefore belong in a comparator table with the publication's provenance. No archived or shifted time series is substituted for course validation. The current input set contains neither blinded file and provides no confirmed fault labels. The December 2025 dates and November 2026 date remain inconsistent in the course brief. [Data dictionary](../docs/data_dictionary.md) · [Source audit](../docs/prior_work_eda.md)


In [ ]:
audit = json.loads((run / 'data_availability.json').read_text())
print(json.dumps(audit, indent=2))
show_files(run, ('audit', 'inventory'), limit=12)


## Interpretation

Review data scope and limitations in the [course report](../docs/final_course_report.md) and [companion guide](../docs/companion_guide.md). The executed values are attributable to the named run and the exact input and checkpoint hashes in its manifest.